# Phase 1: Self-Supervised Pre-Training (SSL)

This notebook implements Phase 1 of the Corponi (E4mer) pipeline using the **unlabelled collection** (127 subjects from `provaforsedef`).

The goal is to learn robust physiological representations via a masked-prediction pretext task before fine-tuning on the Indian dataset.

In [ ]:
import os, sys, shutil

# 1. Environment Setup
WORK_DIR = '/kaggle/working/e4selflearning'
os.chdir('/kaggle/working')

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print("Clonazione repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print("Installazione dipendenze (pinned versions for compatibility)...")
!pip uninstall -y -q numpy scipy pandas
!pip install --no-cache-dir -q "numpy==1.26.4" "scipy==1.12.0" "pandas<2.2" biopsykit flirt ruamel.yaml biosppy hrv-analysis neurokit2 gdown h5py tqdm peakutils lightning torchmetrics distinctipy joypy

print("\nRI-AVVIO IL KERNEL... prosegui dalla cella successiva dopo il riavvio.")
import os
os._exit(0)

Clonazione repository...
Cloning into '/kaggle/working/e4selflearning'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 71 (delta 12), reused 59 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 94.89 KiB | 1.36 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Installazione dipendenze (pinned versions for compatibility)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 957.6 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 71.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 72.4 MB/s eta 0:00:00a 0:00:01


### 2. Loading and Remapping Data (provaforsedef)
We link the preprocessed data and update the metadata to match the local file paths.

In [1]:
import os, glob, shutil, pickle
import numpy as np
print(f"Versione NumPy: {np.__version__} (Deve essere 1.26.4)")

os.chdir('/kaggle/working/e4selflearning')

RAW_UNLABELLED = 'data/preprocessed/unsegmented_unlabelled'
os.makedirs(RAW_UNLABELLED, exist_ok=True)

print("Ricerca dataset 'provaforsedef'...")
meta_files = glob.glob('/kaggle/input/**/provaforsedef/metadata.pkl', recursive=True)
if not meta_files:
    meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)

if not meta_files:
    raise FileNotFoundError("ERRORE: metadata.pkl non trovato!")

src_dir = os.path.dirname(meta_files[0])
print(f"Trovata sorgente in: {src_dir}")

# Link subfolders
for item in os.listdir(src_dir):
    s, d = os.path.join(src_dir, item), os.path.join(RAW_UNLABELLED, item)
    if item == 'metadata.pkl':
        shutil.copy2(s, d)
    else:
        if os.path.exists(d): os.remove(d)
        os.symlink(s, d)

# --- ROBUST REMAPPING ---
print("Aggiornamento percorsi metadata...")
with open(os.path.join(RAW_UNLABELLED, 'metadata.pkl'), 'rb') as f:
    meta = pickle.load(f)

new_sessions_info = {}
found_count = 0

for root, dirs, files in os.walk(RAW_UNLABELLED, followlinks=True):
    if 'channels.h5' in files:
        rel_path = os.path.relpath(root, RAW_UNLABELLED).replace('\\', '/')
        for old_id in meta['sessions_info'].keys():
            if old_id.endswith(rel_path) or rel_path.endswith(old_id):
                new_sessions_info[rel_path] = meta['sessions_info'][old_id]
                found_count += 1
                break

if found_count == 0:
    print("ATTENZIONE: Nessuna sessione mappata! Verifico mappatura di fallback...")
    for root, dirs, files in os.walk(RAW_UNLABELLED, followlinks=True):
        if 'channels.h5' in files:
            rel_path = os.path.relpath(root, RAW_UNLABELLED).replace('\\', '/')
            template_key = list(meta['sessions_info'].keys())[0]
            new_sessions_info[rel_path] = meta['sessions_info'][template_key]
            found_count += 1

meta['sessions_info'] = new_sessions_info
with open(os.path.join(RAW_UNLABELLED, 'metadata.pkl'), 'wb') as f:
    pickle.dump(meta, f)

print(f"Mappatura completata! Sessioni valide trovate: {found_count}")

Versione NumPy: 1.26.4 (Deve essere 1.26.4)
Ricerca dataset 'provaforsedef'...
Trovata sorgente in: /kaggle/input/datasets/ciccinopasticcino7/provaforsedef
Aggiornamento percorsi metadata...
Mappatura completata! Sessioni valide trovate: 111


### 3. Segmentation
Slicing the recordings into windows.

In [2]:
SEG_UNLABELLED = 'data/preprocessed/sl512_ss256_unlabelled'

print("Inizio segmentazione...")
!PYTHONPATH=src python segment.py --data_dir {RAW_UNLABELLED} --output_dir {SEG_UNLABELLED} --segment_length 512 --segmentation_mode 1 --step_size 256 --num_workers 2 --overwrite
print("Segmentazione completata!")

Inizio segmentazione...
2026-04-24 09:53:06.298785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777024386.490049     159 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777024386.542495     159 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777024386.959560     159 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777024386.959607     159 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777024386.959611     159 computation_placer.cc:177

### 4. SSL Pre-Training
Training the encoder via reconstruction loss.

In [3]:
EPOCHS = 50
OUTPUT_DIR = 'output_pretrain'

print("Avvio Pre-Training SSL...")
!PYTHONPATH=src python pre_train.py --dataset {SEG_UNLABELLED} --pretext_task masked_prediction --output_dir {OUTPUT_DIR} --epochs {EPOCHS} --batch_size 32 --e4selflearning --verbose 1

print(f"\nTraining completato. Pesi salvati in: {OUTPUT_DIR}/ckpt_sslearner/model_state.pt")

Avvio Pre-Training SSL...
2026-04-24 09:54:09.450755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777024449.473224     180 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777024449.480731     180 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777024449.501498     180 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777024449.501526     180 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777024449.501533     180 computation_placer.cc:1